# 13 — Temporal Alignment: Static vs. Time-Varying Mediators

Continuation of notebooks 01–06. Checks whether combining static (density, topology) and time-varying (reliability) mediators in one cross-sectional `faircause` estimation is valid, given repeated sensor-window observations from the same physical METR-LA sensors.


In [ ]:
# --- R environment setup for this notebook (safe to re-run; skips if already installed) ---
import subprocess
import sys
subprocess.run(["apt-get", "install", "-y", "-qq", "r-base-core"], stdout=subprocess.DEVNULL)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rpy2"])
get_ipython().run_line_magic("load_ext", "rpy2.ipython")
print("R + rpy2 bridge ready.")


In [ ]:
%%R
#!/usr/bin/env Rscript
# ============================================================================
# 06_temporal_alignment.R
#
# Checks whether combining static (density, topology) and time-varying
# (reliability) mediators in one cross-sectional faircause estimation is
# valid, or whether the panel structure (repeated sensor-windows from the
# same physical sensor) needs explicit handling.
#
# NOT executed/tested locally. Written against documented package APIs
# (sandwich, clubSandwich) — verify on first Colab run.
# ============================================================================

if (!requireNamespace("sandwich", quietly = TRUE)) install.packages("sandwich")
if (!requireNamespace("clubSandwich", quietly = TRUE)) install.packages("clubSandwich")
if (!requireNamespace("lme4", quietly = TRUE)) install.packages("lme4")
library(sandwich)
library(clubSandwich)
library(lme4)

1. Simulate panel data: 207 sensors x multiple time windows each, where
   reliability varies by window but density/topology are fixed per sensor

In [ ]:
%%R
real_data <- read.csv("metr_la_metrics.csv")
n_sensors <- nrow(real_data)
n_windows <- 12

panel_data <- data.frame()

for (i in 1:n_sensors) {
  s_id <- real_data$node_id[i]
  density_s  <- real_data$density[i]
  topology_s <- real_data$topology[i]
  
  base_reliability <- pmax(0, 1.0 - (0.6 * real_data$zero_rate[i] + 0.2 * real_data$cusum_flag_rate[i] + 0.2 * real_data$ewma_flag_rate[i]))
  base_disparity <- real_data$persistence_error[i]
  
  for (w in 1:n_windows) {
    rel_w <- pmin(pmax(rnorm(1, mean = base_reliability, sd = 0.05), 0), 1)
    disp_w <- base_disparity + rnorm(1, sd = base_disparity * 0.1)
    sensor_effect <- 0 # real data captures sensor effect inherently
    
    panel_data <- rbind(panel_data, data.frame(
      sensor_id = s_id,
      window = w,
      density = density_s,
      topology = topology_s,
      reliability = rel_w,
      disparity = disp_w,
      sensor_effect = sensor_effect
    ))
  }
}
cat("Panel data simulated from REAL METR-LA base:", nrow(panel_data), "rows,", n_sensors, "sensors x", n_windows, "windows\n")


2. Naive cross-sectional OLS, IGNORING the panel structure
   (this is what happens if you just feed sensor-windows into faircause
   as if they were independent observations)

In [ ]:
%%R
naive_model <- lm(disparity ~ reliability + topology + density, data = panel_data)
cat("\n=== Naive OLS (ignoring panel structure) ===\n")
print(summary(naive_model)$coefficients)

3. Correct standard errors: cluster by sensor_id
   (does NOT fix bias in the point estimate, only fixes overconfident SEs)

In [ ]:
%%R
clustered_se <- coef_test(naive_model, vcov = "CR2", cluster = panel_data$sensor_id)
cat("\n=== Same model, sensor-clustered standard errors ===\n")
print(clustered_se)

cat("\nCompare the naive SEs above to these clustered SEs. If clustered SEs\n")
cat("are meaningfully larger, the naive cross-sectional approach was\n")
cat("overconfident — its p-values and CIs cannot be trusted as reported.\n")

4. Mixed-effects alternative: explicit random intercept per sensor
   (handles the within-sensor correlation directly, rather than just
   correcting SEs after the fact)

In [ ]:
%%R
mixed_model <- lmer(disparity ~ reliability + topology + density + (1 | sensor_id),
                     data = panel_data)
cat("\n=== Mixed-effects model (random intercept per sensor) ===\n")
print(summary(mixed_model))

5. What this means for faircause specifically

In [ ]:
%%R
cat("\n=== Implications for faircause ===\n")
cat("faircause's fairness_cookbook() does not natively support clustered SEs\n")
cat("or mixed-effects structures — it assumes i.i.d. rows. Three options:\n\n")
cat("Option A (simplest, recommended first pass): collapse each sensor's\n")
cat("  multiple windows into ONE row (e.g. mean reliability across windows)\n")
cat("  before running faircause. Loses the time-varying signal but keeps\n")
cat("  estimation valid and simple.\n\n")
cat("Option B (if time-variation matters): run faircause SEPARATELY per\n")
cat("  window, then pool the resulting Ctf-DE/IE/SE estimates across windows\n")
cat("  using a random-effects meta-analysis (e.g. `metafor` package) rather\n")
cat("  than pooling the raw data.\n\n")
cat("Option C (most rigorous, most work): a full g-computation/marginal\n")
cat("  structural model approach via `gfoRmula` or `ipw`, treating this as\n")
cat("  a genuine longitudinal causal inference problem. Only worth it if\n")
cat("  reliability's time-variation is itself a key part of the story you\n")
cat("  want to tell, not just a nuisance to control for.\n")

6. Effect on Ctf-SE specifically

In [ ]:
%%R
cat("\n=== Effect on the spurious effect (Ctf-SE) component specifically ===\n")
cat("Unmodeled within-sensor correlation acts like an unobserved confounder\n")
cat("shared across a sensor's windows (sensor_effect above). If this\n")
cat("correlates with density (e.g. older/cheaper sensors cluster in\n")
cat("low-density areas AND have a persistent unobserved quality issue),\n")
cat("failing to account for it will typically INFLATE the apparent Ctf-SE\n")
cat("(spurious effect), because some of what looks like 'confounding via C'\n")
cat("is actually 'confounding via unmodeled sensor identity.' This is a\n")
cat("plausible-direction argument, not a proven universal result — the\n")
cat("actual bias direction depends on how sensor_effect correlates with\n")
cat("density in the real data, which should be checked empirically:\n")
cat("  cor(tapply(panel_data$sensor_effect, panel_data$sensor_id, mean),\n")
cat("      tapply(panel_data$density, panel_data$sensor_id, mean))\n")